In [ ]:
import pandas as pd
import warnings
import cudf
import cupy as cp
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

## 5 Cross Validation Sampling

In [ ]:
data_prep = pd.read_csv("data_hasil_preprocessing.csv")
data_prep.head()


,review_detail,case_folded,tokenized,no_stopwords,lemmatized,clean_review
0,"I enjoyed the first season, but I must say I t...",i enjoyed the first season but i must say i th...,"['i', 'enjoyed', 'the', 'first', 'season', 'bu...","['enjoyed', 'first', 'season', 'must', 'say', ...","['enjoyed', 'first', 'season', 'must', 'say', ...",enjoyed first season must say think season eve...
1,I know Iceland is a small country and police d...,i know iceland is a small country and police d...,"['i', 'know', 'iceland', 'is', 'a', 'small', '...","['know', 'iceland', 'small', 'country', 'polic...","['know', 'iceland', 'small', 'country', 'polic...",know iceland small country police thing bit di...
2,"Except K K , no other actor looks comfortable ...",except k k no other actor looks comfortable in...,"['except', 'k', 'k', 'no', 'other', 'actor', '...","['except', 'k', 'k', 'actor', 'looks', 'comfor...","['except', 'k', 'k', 'actor', 'look', 'comfort...",except k k actor look comfortable acting fight...
3,I'm guessing that as a 62 year old white woman...,im guessing that as a year old white woman im ...,"['im', 'guessing', 'that', 'as', 'a', 'year', ...","['im', 'guessing', 'year', 'old', 'white', 'wo...","['im', 'guessing', 'year', 'old', 'white', 'wo...",im guessing year old white woman im target dem...
4,Here's the truth. There's not much to this mov...,heres the truth theres not much to this movie ...,"['heres', 'the', 'truth', 'theres', 'not', 'mu...","['heres', 'truth', 'theres', 'much', 'movie', ...","['here', 'truth', 'there', 'much', 'movie', 's...",here truth there much movie sucked high rating...


In [3]:
data_sampling = pd.read_csv("data_sampling_labelbagging.csv")
data_sampling.head()

,review_id,movie,rating,review_detail,label_komen
0,rw5704482,After Life (2019– ),9.0,"I enjoyed the first season, but I must say I t...",1
1,rw5704483,The Valhalla Murders (2019– ),6.0,I know Iceland is a small country and police d...,0
2,rw5704484,Special OPS (2020– ),7.0,"Except K K , no other actor looks comfortable ...",0
3,rw5704485,#BlackAF (2020– ),8.0,I'm guessing that as a 62 year old white woman...,1
4,rw5704487,The Droving (2020),2.0,Here's the truth. There's not much to this mov...,0


In [4]:
data_sampling["clean_review"] = data_prep["clean_review"]
data_sampling.head()

,review_id,movie,rating,review_detail,label_komen,clean_review
0,rw5704482,After Life (2019– ),9.0,"I enjoyed the first season, but I must say I t...",1,enjoyed first season must say think season eve...
1,rw5704483,The Valhalla Murders (2019– ),6.0,I know Iceland is a small country and police d...,0,know iceland small country police thing bit di...
2,rw5704484,Special OPS (2020– ),7.0,"Except K K , no other actor looks comfortable ...",0,except k k actor look comfortable acting fight...
3,rw5704485,#BlackAF (2020– ),8.0,I'm guessing that as a 62 year old white woman...,1,im guessing year old white woman im target dem...
4,rw5704487,The Droving (2020),2.0,Here's the truth. There's not much to this mov...,0,here truth there much movie sucked high rating...


### Imbalance per Batch

In [5]:
d_data = data_sampling.to_dict(orient="list")

In [6]:
data_5_val = [{"clean_review" : [] , "label_komen" : []},{"clean_review" : [], "label_komen" : []},{"clean_review" : [], "label_komen" : []},{"clean_review" : [], "label_komen" : []},{"clean_review" : [], "label_komen" : []}]

counter = 0

for j in data_5_val :
    for k in range(data_sampling.shape[0] // 5) :
        j["clean_review"].append(d_data["clean_review"][counter])
        j["label_komen"].append(d_data["label_komen"][counter])
        counter+=1
else :
    for k in range(data_sampling.shape[0] % 5) :
        data_5_val[k]["clean_review"].append(d_data["clean_review"][counter])
        data_5_val[k]["label_komen"].append(d_data["label_komen"][counter])
        counter+=1
        
data_5_val

[{'clean_review': ['enjoyed first season must say think season even stronger ricky great job writer actor director brings best superb supporting cast one thing id change id like hear talk less people speak third person pretty hard fault funny yet emotional comedy',
   'know iceland small country police thing bit different europe cmon incompetent police work robs show believability st detective hey got two person interest need talk one could possibly serial killer one visit first nd detective let split',
   'except k k actor look comfortable acting fight scene funny plot repeattative',
   'im guessing year old white woman im target demographic enjoyed show good see others perspective loved wry humor entertaining office park rec make broad negative assumption white though',
   'here truth there much movie sucked high rating overly positive review must submitted cast film crew wont give technical analysis like ameteur movie critic sincere opinionthe droving annoying jump due able hear ope

In [7]:
summ = 0
for i in data_5_val :
    summ += len(i["clean_review"])
summ

295357

In [ ]:
for i in range(len(data_5_val)) :
    data_5_val[i] = cudf.DataFrame(data_5_val[i])

### Balance per Batch

In [9]:
data01terpisah = {"pos" : {"clean_review" : [], "label_komen" : []}, "neg" : {"clean_review" : [], "label_komen" : []}}
summm = {"pos" : 0, "neg" : 0}
for i in range(data_sampling.shape[0]) :
    if d_data["label_komen"][i] == 0 :
        data01terpisah["neg"]["clean_review"].append(d_data["clean_review"][i])
        data01terpisah["neg"]["label_komen"].append(d_data["label_komen"][i])
        summm["neg"] += 1
    else :
        data01terpisah["pos"]["clean_review"].append(d_data["clean_review"][i])
        data01terpisah["pos"]["label_komen"].append(d_data["label_komen"][i])
        summm["pos"] += 1
        
print(summm)

{'pos': 172766, 'neg': 122591}


In [10]:
data_5_val2 = [{"clean_review" : [] , "label_komen" : []},{"clean_review" : [], "label_komen" : []},{"clean_review" : [], "label_komen" : []},{"clean_review" : [], "label_komen" : []},{"clean_review" : [], "label_komen" : []}]

counter = 0
for j in data_5_val2 :
    for k in range(summm["pos"] // 5) :
        j["clean_review"].append(data01terpisah["pos"]["clean_review"][counter])
        j["label_komen"].append(data01terpisah["pos"]["label_komen"][counter])
        counter+=1
else :
    for k in range(summm["pos"] % 5) :
        j["clean_review"].append(data01terpisah["pos"]["clean_review"][counter])
        j["label_komen"].append(data01terpisah["pos"]["label_komen"][counter])
        counter+=1
        
counter = 0
for j in data_5_val2 :
    for k in range(summm["neg"] // 5) :
        j["clean_review"].append(data01terpisah["neg"]["clean_review"][counter])
        j["label_komen"].append(data01terpisah["neg"]["label_komen"][counter])
        counter+=1
else :
    for k in range(summm["neg"] % 5) :
        j["clean_review"].append(data01terpisah["neg"]["clean_review"][counter])
        j["label_komen"].append(data01terpisah["neg"]["label_komen"][counter])
        counter+=1

In [11]:
summ = 0
for i in data_5_val2 :
    summ += len(i["clean_review"])
summ

295357

In [ ]:
for i in range(len(data_5_val2)) :
    data_5_val2[i] = cudf.DataFrame(data_5_val2[i])

## LETSGOOO GRID SEARCHHHH!!!!

In [19]:
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTEENN, SMOTETomek

from cuml.naive_bayes import MultinomialNB
from cuml.linear_model import LogisticRegression
from cuml.svm import LinearSVC
from cuml.ensemble import RandomForestClassifier

model_nb_tuned = MultinomialNB(
    alpha=0.1          # Smoothing parameter: Membantu mengatasi masalah kata yang tidak ada di data training.
                       # Catatan: cuML MultinomialNB belum mendukung parameter fit_prior seperti scikit-learn.
)

model_lr_tuned = LogisticRegression(
    C=2.0,              # Regularisasi: Nilai lebih kecil = regularisasi kuat (mencegah overfitting). 
                        # Nilai lebih besar = model lebih fitting ke data training.
    solver='qn',        # Algoritma optimasi: 'qn' (Quasi-Newton/L-BFGS) adalah default di cuML yang sangat cepat 
                        # memanfaatkan GPU. cuML belum menyediakan solver 'saga'.
    max_iter=1000,      # Jumlah iterasi maksimum: Batas iterasi optimasi. Naikkan jika model belum konvergen.
    linesearch_max_iter=50 # Parameter khusus cuML: Batas iterasi pencarian jalur optimasi untuk menstabilkan solver 'qn'.
)

model_svm_tuned = LinearSVC(
    C=1.0,              # Penalti regularisasi: Mengontrol keseimbangan antara margin yang lebar dan meminimalkan kesalahan klasifikasi.
    loss='squared_hinge', # Fungsi loss: Fungsi penalti standar untuk LinearSVC.
    max_iter=2000,      # Batas iterasi: Ditargetkan agar kalkulasi pada data besar selesai dengan sempurna di GPU.
    fit_intercept=True  # Menentukan apakah bias/intercept perlu dihitung (secara default True di scikit-learn).
                        # Catatan: cuML LinearSVC tidak memerlukan parameter random_state karena sifat optimisasinya di GPU.
)

model_rf_tuned = RandomForestClassifier(
    n_estimators=200,      # Jumlah pohon keputusan: Lebih banyak pohon = lebih stabil dan akurat. Di GPU prosesnya akan tetap instan.
    max_depth=20,          # Kedalaman maksimum pohon: Membatasi kompleksitas pohon agar tidak overfitting dan hemat memori GPU.
    min_samples_split=5,   # Sampel minimum untuk split: Jumlah sampel minimal di sebuah node sebelum dipecah menjadi cabang baru.
    n_bins=15,             # Parameter khusus cuML: Jumlah bin diskritisasi untuk fitur kontinu. cuML menggunakan ini 
                           # untuk mempercepat kalkulasi split di GPU secara signifikan.
    random_state=42        # Konsistensi hasil: Memastikan pembuatan pohon menghasilkan hasil yang sama setiap kali dijalankan.
                           # Catatan: Parameter n_jobs=-1 dihapus karena cuML secara otomatis menggunakan seluruh core paralel GPU.
)

In [ ]:
from cuml.feature_extraction.text import TfidfVectorizer
import heapq as hq
from cuml.metrics import accuracy_score

data_5_batch = {"Imbalance" : data_5_val, "Balance" : data_5_val2}

list_imbalancer = {
    "Tanpa Imbalancer" : None,
    "Smote" : SMOTE(random_state=42),
    "Random Under Sampler" : RandomUnderSampler(random_state=42),
    "Smote Enn" : SMOTEENN(random_state=42),
    "Smote Tomek" : SMOTETomek(random_state=42)
}

tuning_nb = {"alpha" : [0.01, 0.1, 0.5, 1.0, 5.0, 10.0]}
tuning_svm = {"C": [], "loss" : [], "max_iter" : [], "fit_intercept" : []}
tuning_rf = {"n_estimators" : [], "max_depth" : [], "min_samples_split" : [], "n_bins" : []}
tuning_lr = {"C" : [], "solver" : [], "max_iter" : [], "linesearch_max_iter" : []}

qiu_primer = []
counter = 0
for h in data_5_batch:
    paket_data = data_5_batch[h]
    
    for i in range(len(paket_data)):
        train = cudf.DataFrame({"clean_review" : [], "label_komen" : []})
        test = cudf.DataFrame({"clean_review" : [], "label_komen" : []})
        batch_data_test = i
        
        for j in range(len(paket_data)):
            if j == i:
                test = cudf.concat([test, paket_data[j]], axis=0, ignore_index=True)
            else:
                train = cudf.concat([train, paket_data[j]], axis=0, ignore_index=True)
        
        train['clean_review'] = train['clean_review'].fillna('')
        test['clean_review'] = test['clean_review'].fillna('')
        
        tfidf_vectorizer = TfidfVectorizer(
            min_df=5,
            max_df=0.8,
            max_features=10000
        )
        
        x_train = tfidf_vectorizer.fit_transform(train['clean_review'])
        x_test = tfidf_vectorizer.transform(test["clean_review"])
        
        y_train = train["label_komen"]
        y_test = test["label_komen"]
        
        for j in list_imbalancer:
            imbalancer_name = j
            
            if j != "Tanpa Imbalancer":
                x_train_dense = x_train.toarray()
                y_train_cpu = y_train.to_pandas() 

                x_train, y_train = list_imbalancer[j].fit_resample(x_train_dense, y_train_cpu)
                x_train = cp.array(x_train)
                y_train = cudf.Series(y_train)
            
            qiu_sekunder = []
            for k in tuning_nb["alpha"] :
                model = MultinomialNB(alpha= k)
                model.fit(x_train, y_train)
                y_pred = model.predict(x_test)
                akurasi = accuracy_score(y_test, y_pred)
                hq.heappush(qiu_sekunder,(-akurasi,counter,model,f"MultinomialNB(alpha = {k})"))
                counter += 1
            hasil = hq.heappop(qiu_sekunder)
            model_nb = (-hasil[0],hasil[2],hasil[3])
            y_pred_train_nb = hasil[2].predict(x_train)
            y_pred_test_nb = hasil[2].predict(x_test)
            
            qiu_sekunder = []
            for k in tuning_svm["C"] :
                for l in tuning_svm["loss"] :
                    for m in tuning_svm["max_iter"] :
                        for n in tuning_svm["fit_intercept"] :
                            model = LinearSVC(C = k, loss = l, max_iter = m, fit_intercept = n)
                            model.fit(x_train, y_train)
                            y_pred = model.predict(x_test)
                            akurasi = accuracy_score(y_test, y_pred)
                            hq.heappush(qiu_sekunder,(-akurasi,counter,model,f"LinearSVC(C = {k}, loss = {l}, max_iter = {m}, fit_intercept = {n})"))
                            counter += 1
            hasil = hq.heappop(qiu_sekunder)
            model_svc = (-hasil[0],hasil[2],hasil[3])
            y_pred_train_svc = hasil[2].predict(x_train)
            y_pred_test_svc = hasil[2].predict(x_test)
            
            qiu_sekunder = []
            for k in tuning_rf["n_estimators"] :
                for l in tuning_rf["max_depth"] :
                    for m in tuning_rf["min_samples_split"] :
                        for n in tuning_rf["n_bins"] :
                            model = RandomForestClassifier(n_estimators = k, max_depth = l, min_samples_split = m, n_bins = n)
                            model.fit(x_train, y_train)
                            y_pred = model.predict(x_test)
                            akurasi = accuracy_score(y_test, y_pred)
                            hq.heappush(qiu_sekunder,(-akurasi,counter,model,f"RandomForestClassifier(n_estimators = {k}, max_depth = {l}, min_samples_split = {m}, n_bins = {n})"))
                            counter += 1
            hasil = hq.heappop(qiu_sekunder)
            model_rf = (-hasil[0],hasil[2],hasil[3])
            y_pred_train_rf = hasil[2].predict(x_train)
            y_pred_test_rf = hasil[2].predict(x_test)
            
            
            
            
            
            
            


            
                
                
            
            
            
        
    
    
            

5 Cross Validation Sampling V

pecah jadi Train dan Test V

tf idf

imbalancer

training base model

training meta model